# **一鍵生成全班段考成績圖表 (v2)**

v2 更新 (2026-11):
- 相容 pandas 3: 以 `df.iloc[列, 欄]` 取代 `df.iloc[列][欄]`
- 中文字型改用免費開源的 Noto Sans TC (思源黑體繁體中文版)
- 修正 PDF 報告中「師生學習輔導與省思」標題與方框重疊


# **1. 安裝需要的python軟體套件**

In [ ]:

%pip install matplotlib pandas seaborn scipy pycirclize fpdf2 pytictoc openpyxl


# **2. 載入需要的python軟體套件**

In [ ]:

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import csv
import pandas as pd
import seaborn as sns
import os
import subprocess
import warnings

from matplotlib.colors import LinearSegmentedColormap
from scipy.optimize import curve_fit
from math import pi
from pycirclize import Circos
from fpdf import FPDF
from pytictoc import TicToc


## 屏蔽執行時出現無關痛癢的警告訊息
warnings.filterwarnings('ignore')


# **3. 下載與設定中文字型**

In [ ]:

## 檢查是否已下載中文字型: Noto Sans TC（思源黑體繁體中文版, 免費開源字型）
local_font_path = './NotoSansTC-Regular.otf'

if ( os.path.exists(local_font_path) == False):
    ## 從 Google 官方的 Noto CJK 字型庫下載
    !wget -q https://github.com/notofonts/noto-cjk/raw/main/Sans/SubsetOTF/TC/NotoSansTC-Regular.otf

## 指定字型
matplotlib.font_manager.fontManager.addfont(local_font_path)
matplotlib.rc('font', family='Noto Sans TC')


# **4. 檢查是否已有成績資料excel檔案**

In [ ]:

## 使用自己的成績資料: 先把 excel 檔上傳到左側「檔案」區, 再把下面的檔名改成你的檔名 (例如 './my_class.xlsx')
## 範例成績資料檔 (學生皆為虛構人物)
local_data_path = './demo.xlsx'

if ( os.path.exists(local_data_path) == False):
    ## 從 GitHub 下載範例成績
    !wget -q https://raw.githubusercontent.com/raycc4p53/exam-report/main/demo.xlsx -O demo.xlsx


# **5. 設定輸出資料夾路徑**

In [ ]:

## 輸出資料夾--根目錄路徑
out_chart_dir = "./out_charts"

## 輸出資料夾--子目錄名稱
out_chart_subdir = [ "01_chinese", "02_english", "03_social", "04_math", "05_science", "06_avg", "07_ranking", "08_radar", "09_pdf", "10_jpg" ]

## 檢查是否已有根目錄
if (os.path.exists(out_chart_dir) ==False):
    ## 建立根目錄
    os.mkdir(out_chart_dir)

## 檢查是否已有子目錄
for i in out_chart_subdir:
    ## 子目錄路徑
    out_dir = out_chart_dir + "/" + i

    if (os.path.exists(out_dir)==False):
        ## 建立子目錄
        os.mkdir(out_dir)


# **6 . 載入副程式#1 (plot_subject_chronicle): 繪製歷年成績比較圖**

In [ ]:
def plot_subject_chronicle(subject_name, df, df2, image_dir):

    ## 學期數
    n_semester= df.shape[1]-2

    ## 學生總數
    n_student = df.shape[0]

    ## 段考次數
    n_exam = df.shape[1]-2

    df_avg = df2.copy()
    df_avg.set_index(df_avg.columns[0], inplace=True)

    for i in range(0,n_student,1):

        ##
        a = df.iloc[i, 2:n_exam+2]
        df_avg.loc[:, df.iloc[i]['學生']] = a

        ###### 畫出三條歷年單科成績曲線#########
        fig, ax = plt.subplots()

        ax.plot(df_avg.iloc[:,2],'bo-', linewidth=3, markersize=6) #歷年學生段考成績
        ax.plot(df_avg.iloc[:,1], 'r^--', linewidth=1.5, markersize=4) #歷年全校高標
        ax.plot(df_avg.iloc[:,0], 'gs:',linewidth=1.5, markersize=4) #歷年全校平均

        ## 設定輸出圖形尺寸
        fig.set_size_inches(8, 6)

        ## 設定y軸座標、標題
        plt.ylim(0,100)
        plt.ylabel('成績',fontsize=12)

        ## 圖表標題: 學科名
        chart_title =   subject_name
        plt.title( chart_title,fontsize=18)

        ## 畫出一條60分及格水平線
        plt.axhline(y = 60, color = 'gray', linestyle = 'solid')
        plt.legend([df_avg.columns[2], df_avg.columns[1], df_avg.columns[0]], loc='lower left')

        ## 加上x軸格線
        plt.grid(axis='x')

        xx = list(range(0, n_exam))
        yy = a[0:n_exam]

        ## 在圖中標注學生學科成績
        for x, y in zip(xx, yy):
            ax.text(x-0.1, y-4, f"{y:g}", fontsize=9)

        ## 設定輸出jpg圖檔檔名
        fn = image_dir + str(df.iloc[i]['座號'])  +  ".jpg"

        ## 設定解析度並存檔
        plt.savefig(fn,dpi=200)

        df_avg.drop(df_avg.columns[[-1]], axis=1, inplace=True)

        plt.close(fig)


# **7. 載入副程式#2 (plot_avg_chronicle): 繪製歷年平均比較圖**

In [ ]:
def plot_avg_chronicle(df, df2,image_dir):

    subject_name = "平均"

    ## 學期數
    n_semester= df.shape[1]-2

    ## 學生總數
    n_student = df.shape[0]

    ## 段考次數
    n_exam = df.shape[1]-2
    df_avg = df2.copy()

    #
    df_avg = df_avg.drop(['學年', '學期', '段考','班級'], axis=1)
    df_avg.set_index(df_avg.columns[0], inplace=True)

    for i in range(0,n_student,1):

        ##
        a = df.iloc[i, 2:n_exam+2]
        df_avg.loc[:, df.iloc[i]['學生']] = a

        ###### 畫出兩條歷年平均成績曲線#########
        fig, ax = plt.subplots()

        ax.plot(df_avg.iloc[:,1],'bo-', linewidth=3, markersize=6) #歷年學生平均分數
        ax.plot(df_avg.iloc[:,0], 'gs--',linewidth=1.5, markersize=4) #歷年全校平均分數

        ## 設定輸出圖形尺寸
        fig.set_size_inches(8, 6)

        ## 設定y軸座標、標題
        plt.ylim(0,100)
        plt.ylabel('成績',fontsize=12)

        ## 圖表標題: 平均
        chart_title =   subject_name
        plt.title( chart_title,fontsize=18)

        ## 畫出一條60分及格水平線
        plt.axhline(y = 60, color = 'gray', linestyle = 'solid')
        plt.legend([df_avg.columns[1], df_avg.columns[0]], loc='lower left')

        ## 加上x軸格線
        plt.grid(axis='x')
        xx = list(range(0, n_exam))
        yy = a[0:n_exam]

        ## 在圖中標注學生平均成績
        for x, y in zip(xx, yy):
            ax.text(x-0.1, y-4, f"{y:g}", fontsize=9)

        ## 設定輸出jpg圖檔檔名
        fn = image_dir + str(df.iloc[i]['座號'])  +  ".jpg"

        ## 設定解析度並存檔
        plt.savefig(fn,dpi=200)

        df_avg.drop(df_avg.columns[[-1]], axis=1, inplace=True)

        plt.close(fig)


# **8. 載入副程式#3 (plot_ranking_chronicle): 繪製歷年排名比較圖**

In [ ]:

def plot_ranking_chronicle(df, image_dir):

    ## 學期數
    n_semester= df.shape[1]-2

    ## 學生總數
    n_student = df.shape[0]

    ## 段考次數
    n_exam = df.shape[1]-2

    for i in range(0,n_student,1):

    #### 畫出ㄧ條歷年校排名曲線
        fig, ax = plt.subplots()

        ##
        a = df.iloc[i, 2:n_exam+2]

        ax.plot(a,'bo-', linewidth=3, markersize=6,label=df.iloc[i]['學生']) ## 歷年學生校排名

        ## 設定輸出圖形尺寸
        fig.set_size_inches(8, 6)

        ## 設定y軸座標、標題
        ## 全校假設為355人
        plt.ylim(0,355)

        ## 將y軸座標反轉,視覺呈現較為直覺: 排名越高(數值越小), 在y軸的位置越高
        ax.invert_yaxis()

        plt.ylabel('排名',fontsize=12)

        ## 圖表標題: 平均
        chart_title = "校排名"

        plt.title( chart_title,fontsize=18)

        ## 根據全年級學生總數,畫出兩條直線 ,可清楚區別前(1/3),中(2/3),後段排名
        ## 範例的學生人數355, 因此以100,200來區隔

        plt.axhline(y = 100, color = 'gray', linestyle = 'solid')
        plt.axhline(y = 200, color = 'gray', linestyle = 'solid')

        xx = list(range(0, n_exam))
        yy = a[0:n_exam]

        ## 在圖中標注學生排名數值
        for x, y in zip(xx, yy):
                    ax.text(x, y+15, f"{y:g}", fontsize=9)

        #### 產生一條回歸曲線

        ##
        df2 = pd.DataFrame({'x': xx,'y': yy})

        df2 = df2.astype({'x': int, 'y': int})

        ##設定四次方回歸
        polyline = np.linspace(0, n_semester-1, n_semester)
        model4 = np.poly1d(np.polyfit(df2.x, df2.y, 4))

        a2 = pd.Series(model4(polyline))

        ax.plot(a2,'--',linewidth=1.5,color='red', label = '趨勢 (回歸曲線)') # 回歸曲線

        ax.grid(axis='x')

        ## 如果學生排名在270之後, legend的位置在圖左上方. 反之則在左下方
        if a.iloc[1]<270 :
            plt.legend( loc='lower left',edgecolor="gray")
        else:
            plt.legend( loc='upper left',edgecolor="gray")

        ## 設定輸出jpg圖檔檔名
        fn = image_dir + str(df.iloc[i]['座號']) + ".jpg"

        ## 設定解析度並存檔
        plt.savefig(fn,dpi=200)

        plt.close(fig)


# **9. 開始執行: 啟動計時碼表**

###################### start running ######################


In [ ]:

## 開始計時
t = TicToc()
t.tic()


# **10. 讀取學生成績excel檔案**

In [ ]:

#### 指定excel檔名

excel_fn = local_data_path

## 取得excel內全部表單(sheet)名稱
excel_sheet_list = pd.ExcelFile(excel_fn)

## 讀取excel內全部表單
excel_all_sheets = pd.read_excel(excel_fn,sheet_name=None,header=0)

## 將每個表單指定為獨立的dataframe

## 1. 國
excel_score_chinese_df =  excel_all_sheets[excel_sheet_list.sheet_names[0]]

## 2. 英
excel_score_english_df = excel_all_sheets[excel_sheet_list.sheet_names[1]]

## 3. 社
excel_score_social_df =excel_all_sheets[excel_sheet_list.sheet_names[2]]

## 4. 數
excel_score_math_df = excel_all_sheets[excel_sheet_list.sheet_names[3]]

## 5. 自
excel_score_science_df = excel_all_sheets[excel_sheet_list.sheet_names[4]]

## 6. 均
excel_score_avg_df = excel_all_sheets[excel_sheet_list.sheet_names[5]]

## 7. 班排
excel_class_ranking_df = excel_all_sheets[excel_sheet_list.sheet_names[6]]

## 8. 校排
excel_school_ranking_df = excel_all_sheets[excel_sheet_list.sheet_names[7]]

## 9. 校均
excel_school_avg_df = excel_all_sheets[excel_sheet_list.sheet_names[8]]

## 10. 國_校標
excel_avg_chinese_df = excel_all_sheets[excel_sheet_list.sheet_names[9]]

## 11. 英_校標
excel_avg_english_df = excel_all_sheets[excel_sheet_list.sheet_names[10]]

## 12. 社_校標
excel_avg_social_df = excel_all_sheets[excel_sheet_list.sheet_names[11]]

## 13. 數_校標
excel_avg_math_df = excel_all_sheets[excel_sheet_list.sheet_names[12]]

## 14. 自_校標
excel_avg_science_df = excel_all_sheets[excel_sheet_list.sheet_names[13]]


# **11. 依序畫出全班每位學生的歷年(國,英,社,數,自)成績比較圖**

In [ ]:

#### 國

## 設定輸出資料夾路徑
out_tmp = out_chart_dir + "/" + out_chart_subdir[0]+"/"

## 執行副程式(plot_subject_chronicle)畫圖
plot_subject_chronicle("國文",excel_score_chinese_df,excel_avg_chinese_df, out_tmp)

#### 英

## 設定輸出資料夾路徑
out_tmp = out_chart_dir + "/" + out_chart_subdir[1]+"/"

## 執行副程式(plot_subject_chronicle)畫圖
plot_subject_chronicle("英語",excel_score_english_df,excel_avg_english_df, out_tmp)

#### 社

## 設定輸出資料夾路徑
out_tmp = out_chart_dir + "/" + out_chart_subdir[2]+"/"

## 執行副程式(plot_subject_chronicle)畫圖
plot_subject_chronicle("社會",excel_score_social_df,excel_avg_social_df, out_tmp)

#### 數

## 設定輸出資料夾路徑
out_tmp = out_chart_dir + "/" + out_chart_subdir[3]+"/"

## 執行副程式(plot_subject_chronicle)畫圖
plot_subject_chronicle("數學",excel_score_math_df,excel_avg_math_df, out_tmp)

#### 自

## 設定輸出資料夾路徑
out_tmp = out_chart_dir + "/" + out_chart_subdir[4]+"/"

## 執行副程式(plot_subject_chronicle)畫圖
plot_subject_chronicle("自然",excel_score_science_df,excel_avg_science_df, out_tmp)


# **12. 畫出全班每位學生的歷年平均比較圖**

In [ ]:

#### 歷年全校平均

## 設定輸出資料夾路徑
out_tmp = out_chart_dir + "/" + out_chart_subdir[5]+"/"

## 執行副程式(plot_avg_chronicle)畫圖
plot_avg_chronicle(excel_score_avg_df,excel_school_avg_df,out_tmp)


# **13. 畫出全班每位學生的歷年排名比較圖**

In [ ]:

## 設定輸出資料夾路徑
out_tmp = out_chart_dir + "/" + out_chart_subdir[6]+"/"

## 執行副程式(plot_ranking_chronicle)畫圖
plot_ranking_chronicle(excel_school_ranking_df, out_tmp)


# **14. 畫出全班每位學生的成績雷達圖(學科優勢傾向)**

In [ ]:
#### 取得最近一次段考的各科全校平均成績

## 全班人數
n_student= excel_score_chinese_df.shape[0]

## 最近一次月考成績的欄位是在excel的第幾列
col_latest_exam = excel_score_chinese_df.shape[1]-1

## 最近一次月考的全校平均是在excel的第幾行
row_latest_exam = excel_avg_chinese_df.shape[0]-1

## 最近一次段考全校國文平均分數
avg_chinese_latest = excel_avg_chinese_df.iloc[row_latest_exam, 1]

## 最近一次段考全校英語平均分數
avg_english_latest = excel_avg_english_df.iloc[row_latest_exam, 1]

## 最近一次段考全校社會平均分數
avg_social_latest = excel_avg_social_df.iloc[row_latest_exam, 1]

## 最近一次段考全校數學平均分數
avg_math_latest = excel_avg_math_df.iloc[row_latest_exam, 1]

## 最近一次段考全校自然平均分數
avg_science_latest = excel_avg_science_df.iloc[row_latest_exam, 1]

#print(avg_chinese_latest,avg_english_latest,avg_social_latest,avg_math_latest,avg_science_latest)

## 設定輸出資料夾路徑
out_tmp = out_chart_dir + "/" + out_chart_subdir[7]+"/"

for i in range(0,n_student,1):

    ## 學生座號
    student_no = excel_score_chinese_df.iloc[i, 0]

    ## 學生姓名
    student_name = excel_score_chinese_df.iloc[i, 1]

    ## 最近一次段考的各科成績

    ## 國
    chinese_latest = excel_score_chinese_df.iloc[i, col_latest_exam]

    ## 英
    english_latest = excel_score_english_df.iloc[i, col_latest_exam]

    ## 社
    if (excel_score_social_df.iloc[i, col_latest_exam] % round(excel_score_social_df.iloc[i, col_latest_exam]) ==0):

        social_latest = round(excel_score_social_df.iloc[i, col_latest_exam])
    else:
        social_latest = excel_score_social_df.iloc[i, col_latest_exam]

    ## 數
    math_latest = excel_score_math_df.iloc[i, col_latest_exam]

    ## 自
    if (excel_score_science_df.iloc[i, col_latest_exam] % round(excel_score_science_df.iloc[i, col_latest_exam]) ==0):

        science_latest = round(excel_score_science_df.iloc[i, col_latest_exam])
    else:
        science_latest = excel_score_science_df.iloc[i, col_latest_exam]

    # 五科總分
    sum_latest = chinese_latest + english_latest + social_latest + math_latest + science_latest

    # 五科平均
    avg_latest = excel_score_avg_df.iloc[i, col_latest_exam]

    #print(chinese_latest, english_latest, social_latest, math_latest, science_latest,sum_latest,avg_latest,class_ranking_latest,school_ranking_latest)

    ## 將各科學生成績,全校平均帶入用來產生雷達圖的dataframe

    dfx = pd.DataFrame(
        data=[
            [chinese_latest,english_latest,social_latest,math_latest,science_latest],
            [avg_chinese_latest,avg_english_latest,avg_social_latest,avg_math_latest,avg_science_latest]
        ],

        index=[student_name,"校平均"],
        columns=["國文", "英語", "社會", "數學","自然" ],
    )

    ## 雷達圖基本設定
    circos = Circos.radar_chart(
        dfx,
        fill=True,
        marker_size=6,
        vmax=100,
        grid_interval_ratio=0.2,
    )

    ## 雷達圖標題
    circos.text("學科優勢傾向", r=125, size=15, weight="bold")

    ## 產生雷達圖
    fig = circos.plotfig()
    _ = circos.ax.legend(loc="upper right",fontsize=14)

    ## 設定雷達圖輸出圖檔路徑
    radar_fn = out_tmp + str(student_no) + '.jpg'

    ## 存檔
    fig.savefig(radar_fn)
    plt.close(fig)


# **15. 產生全班每位學生的PDF完整報告**

In [ ]:
#### 取得最近一次段考的資訊與各科成績

## 學年度 (112)
acad_year = excel_school_avg_df.iloc[-1, 0]

## 學期 (下)
acad_term = excel_school_avg_df.iloc[-1, 1]

## 段考 (3)
acad_exam = excel_school_avg_df.iloc[-1, 2]

## 班級名稱 (800)
acad_class = excel_school_avg_df.iloc[-1, 4]

## PDF成績單報表主標題： #112學年下學期: 第3次段考
report_title = str(acad_year) + "學年" + str(acad_term) + "學期: 第" + str(acad_exam) + "次段考"

print(report_title)

#### 學生成績

## 全班學生人數
n_student= excel_score_chinese_df.shape[0]

## 最近一次月考成績的欄位是在excel的第幾列
col_latest_exam = excel_score_chinese_df.shape[1]-1

## 最近一次月考的全校平均是在excel的第幾行
row_latest_exam = excel_avg_chinese_df.shape[0]-1

## 設定輸出資料夾路徑
out_tmp = out_chart_dir + "/" + out_chart_subdir[8]+"/"

#### 先前步驟已產生圖檔的路徑

## 國
image_chinese = out_chart_dir + "/" + out_chart_subdir[0]+"/"

## 英
image_english = out_chart_dir + "/" + out_chart_subdir[1]+"/"

## 社
image_social = out_chart_dir + "/" + out_chart_subdir[2]+"/"

## 數
image_math = out_chart_dir + "/" + out_chart_subdir[3]+"/"

## 自
image_science = out_chart_dir + "/" + out_chart_subdir[4]+"/"

## 平均
image_avg = out_chart_dir + "/" + out_chart_subdir[5]+"/"

## 校排
image_ranking = out_chart_dir + "/" + out_chart_subdir[6]+"/"

## 雷達圖
image_radar = out_chart_dir + "/" + out_chart_subdir[7]+"/"

## PDF報告
image_pdf = out_chart_dir + "/" + out_chart_subdir[8]+"/"


#### 設定全班PDF版型與尺寸
pdf_all = FPDF(orientation="P", unit="mm", format="A4",)

#### 產生個別學生的PDF報告
for i in range(0,n_student,1):

    ## 學生座號
    student_no = excel_score_chinese_df.iloc[i, 0]

    ## 學生姓名
    student_name = excel_score_chinese_df.iloc[i, 1]

    ## PDF成績單報表副標題： 學生資訊
    report_title2 = str(acad_class) + "-" + str(student_no ) + " " + student_name

    ## 取得最近一次各科段考成績

    ## 國
    chinese_latest = excel_score_chinese_df.iloc[i, col_latest_exam]

    ## 英
    english_latest = excel_score_english_df.iloc[i, col_latest_exam]

    ## 社
    if (excel_score_social_df.iloc[i, col_latest_exam] % round(excel_score_social_df.iloc[i, col_latest_exam]) ==0):

        social_latest = round(excel_score_social_df.iloc[i, col_latest_exam])
    else:
        social_latest = excel_score_social_df.iloc[i, col_latest_exam]

    ## 數
    math_latest = excel_score_math_df.iloc[i, col_latest_exam]

    ## 自
    if (excel_score_science_df.iloc[i, col_latest_exam] % round(excel_score_science_df.iloc[i, col_latest_exam]) ==0):

        science_latest = round(excel_score_science_df.iloc[i, col_latest_exam])
    else:
        science_latest = excel_score_science_df.iloc[i, col_latest_exam]

    ## 五科總分
    sum_latest = chinese_latest + english_latest + social_latest + math_latest + science_latest

    ## 五科平均
    avg_latest = excel_score_avg_df.iloc[i, col_latest_exam]

    ## 班排名
    class_ranking_latest = excel_class_ranking_df.iloc[i, col_latest_exam]

    ## 校排名
    school_ranking_latest = excel_school_ranking_df.iloc[i, col_latest_exam]

    ## 設定個人PDF版型與尺寸
    pdf = FPDF(orientation="P", unit="mm", format="A4",)

    #### 接下來的每個步驟,都要做兩次. 一次是給全班PDF,另一次給個人PDF

    ## 產生一頁空白PDF for 個人
    pdf.add_page()

    ## 產生一頁空白PDF for 全班
    pdf_all.add_page()

    # 設定中文字型
    pdf.add_font('Noto Sans TC', '', local_font_path)

    pdf_all.add_font('Noto Sans TC', '', local_font_path)

    ## 寫入主標題: 段考資訊
    pdf.set_font('Noto Sans TC', size=20)
    pdf.cell(200, 10, text = report_title, ln=1, align = 'C')

    pdf_all.set_font('Noto Sans TC', size=20)
    pdf_all.cell(200, 10, text = report_title, ln=1, align = 'C')

    ## 寫入副標題: 學生資訊
    pdf.set_font('Noto Sans TC', size=20)
    pdf.cell(200, 10, text = report_title2, ln=1, align = 'C')

    pdf_all.set_font('Noto Sans TC', size=20)
    pdf_all.cell(200, 10, text = report_title2, ln=1, align = 'C')

    ## 畫一條分隔線
    pdf.line(5, 33, 205, 33)

    pdf_all.line(5, 33, 205, 33)

    # 設定成績table中文字型
    pdf.set_font('Noto Sans TC', size=16)

    pdf_all.set_font('Noto Sans TC', size=16)

    #print(chinese_latest, english_latest, social_latest, math_latest, science_latest,sum_latest,avg_latest,class_ranking_latest,school_ranking_latest)

    #### 將學生成績填入table
    dfx2 = pd.DataFrame(
        data=[
            [chinese_latest, english_latest, social_latest,	math_latest,	science_latest,sum_latest,avg_latest,class_ranking_latest,school_ranking_latest],
        ],

        columns=["國文", "英語", "社會","數學", "自然","總分", "平均","班排","校排"],
    )

    pdf.cell(60)
    pdf_all.cell(60)

    pdf.cell(100, 10, '', 0, 2, 'C')
    pdf.cell(-55)

    pdf_all.cell(100, 10, '', 0, 2, 'C')
    pdf_all.cell(-55)

    columnNameList = list(dfx2.columns)

    #### 產生一個各科成績表格

    ## 填入表格欄位標題
    for header in columnNameList[:-1]:
        pdf.cell(20, 8, header, 0, 0, 'C')
        pdf_all.cell(20, 8, header, 0, 0, 'C')

    pdf.cell(20, 8, columnNameList[-1], 0, 2, 'C')
    pdf.cell(-160)

    pdf_all.cell(20, 8, columnNameList[-1], 0, 2, 'C')
    pdf_all.cell(-160)

    ## 填入各科成績
    for row in range(0, len(dfx2)):
        for col_num, col_name in enumerate(columnNameList):
            pdf.cell(20,8, str(dfx2['%s' % (col_name)].iloc[row]), 1, 0 ,'C')

            pdf_all.cell(20,8, str(dfx2['%s' % (col_name)].iloc[row]), 1, 0 ,'C')

    ## 先前已產生的圖檔,在不同資料夾裡面的檔案命名都是: 學生座號.jpg
    general_fn = str(student_no) + '.jpg'

    ## 個人pdf報告檔案命名: 學生座號.pdf
    pdf_fn = str(student_no) + '.pdf'

    #### 寫入第一行: 插入國,英,社,三張歷年成績比較圖

    pdf.image(image_chinese + general_fn, x= 10,y=60,h=50,keep_aspect_ratio=True)
    pdf.image(image_english + general_fn, x= 72,y=60,h=50,keep_aspect_ratio=True)
    pdf.image(image_social + general_fn, x= 134,y=60,h=50,keep_aspect_ratio=True)

    pdf_all.image(image_chinese + general_fn, x= 10,y=60,h=50,keep_aspect_ratio=True)
    pdf_all.image(image_english + general_fn, x= 72,y=60,h=50,keep_aspect_ratio=True)
    pdf_all.image(image_social + general_fn, x= 134,y=60,h=50,keep_aspect_ratio=True)

    ## 寫入第二行: 插入數,自,平均 三張歷年成績比較圖
    pdf.image(image_math + general_fn, x= 10,y=108,h=50,keep_aspect_ratio=True)
    pdf.image(image_science + general_fn, x= 72,y=108,h=50,keep_aspect_ratio=True)
    pdf.image(image_avg + general_fn, x= 134,y=108,h=50,keep_aspect_ratio=True)

    pdf_all.image(image_math + general_fn, x= 10,y=108,h=50,keep_aspect_ratio=True)
    pdf_all.image(image_science + general_fn, x= 72,y=108,h=50,keep_aspect_ratio=True)
    pdf_all.image(image_avg + general_fn, x= 134,y=108,h=50,keep_aspect_ratio=True)

    ## 寫入第三行: 插入雷達圖, 歷年校排比較圖
    pdf.image(image_radar + general_fn, x= 20,y=155,w=75,keep_aspect_ratio=True)
    pdf.image(image_ranking + general_fn, x= 110,y=155,w=90,keep_aspect_ratio=True)

    pdf_all.image(image_radar + general_fn, x= 20,y=155,w=75,keep_aspect_ratio=True)
    pdf_all.image(image_ranking + general_fn, x= 110,y=155,w=90,keep_aspect_ratio=True)

    ## 產生一個空白的矩形box ,讓師生雙方都可以手寫評論或心得
    ## 標題放在方框上方 (y=238~248), 避免與方框 (y=250) 重疊
    pdf.set_xy(5, 238)
    pdf.set_font('Noto Sans TC', size=14)
    pdf.cell(200, 10, text = "師生學習輔導與省思", ln=1, align = 'C')

    ## 標題放在方框上方 (y=238~248), 避免與方框 (y=250) 重疊
    pdf_all.set_xy(5, 238)
    pdf_all.set_font('Noto Sans TC', size=14)
    pdf_all.cell(200, 10, text = "師生學習輔導與省思", ln=1, align = 'C')

    y = 250

    pdf.set_draw_color(100)
    pdf.rect(15, y, 180, 35, round_corners=True, style="D")

    pdf_all.set_draw_color(100)
    pdf_all.rect(15, y, 180, 35, round_corners=True, style="D")

    ## 存檔: 個人PDF
    pdf.output(out_tmp + pdf_fn)

## 存檔: 全班PDF 檔名: all_pdf.pdf
pdf_all.output(out_chart_dir + "/"+ "all_pdf.pdf" )


***

##############################

# **PDF 轉 JPG圖檔**

##############################

***

# **16. 安裝開源影音轉檔軟體: imagemagick**

imagemagick並不是python的軟體套件

試過幾個可以把PDF轉圖檔的python套件之後,都不太滿意

Google Colab的runtime其實就是一台安裝了ubuntu linux 22.4的虛擬電腦

所以就直接安裝超強的轉檔軟體imagemagick, 利用它的convert指令,輕鬆的將PDF轉成JPG圖檔

In [ ]:

!apt install imagemagick


# **17. 刪除檔案: policy.xml**

如果要將PDF轉為圖檔的話,必須先修改/etc/ImageMagick-6/policy.xml裡面的某個設定

不然就無法使用convert指令轉檔

我們就使用偷吃步,免修改,直接把policy.xml刪掉即可


In [ ]:

## 秀出刪除前,資料夾的檔案列表
!ls -l /etc/ImageMagick-6/

!rm -f /etc/ImageMagick-6/policy.xml

## 秀出刪除後,資料夾的檔案列表
!ls -l /etc/ImageMagick-6/


# **18. 開始將PDF轉成JPG圖檔!!**

In [ ]:

## 設定輸入PDF資料夾路徑
pdf_dir = out_chart_dir + "/" + out_chart_subdir[8]+"/"

## 設定輸出JPG資料夾路徑
jpg_dir = out_chart_dir + "/" + out_chart_subdir[9]+"/"

## 將PDF資料夾內的檔案,根據檔名以數字升冪排序
dir_list = sorted(os.listdir(pdf_dir))

n_file = len(dir_list)

for i in range(0,n_file,1):

    ## PDF 輸入檔名
    fn1 = pdf_dir + dir_list[i]

    ## JPG 輸出檔名
    fn2 = jpg_dir + dir_list[i].replace(".pdf",".jpg")

    ## 產生完整的convert指令
    cmd = "convert -density 300 " + fn1 + " -quality 100 " + fn2

    ## 執行轉檔
    subprocess.run(cmd, shell=True)


###################### end running ######################

# **19. 執行完畢: 停止計時碼表**

In [ ]:
t.toc()

# **20.（選用）掛載你的 Google Drive**

想把全部結果存到自己的 Google Drive，請等上面的 PDF 和 JPG 圖檔都產生完畢，再做下面兩個步驟。

1. 把下面這格程式碼**每一行開頭的 `#` 刪掉**（也可以選取全部程式碼，按 `Ctrl + /`；Mac 按 `⌘ + /`）
2. 按這格左邊的 ▶ 執行
3. Colab 會跳出授權視窗，選擇你的 Google 帳號，按「允許」，讓 Colab 可以存取你的 Google Drive


In [ ]:

#from google.colab import drive
#drive.mount('/content/drive')


# **21.（選用）將全部輸出檔案複製到你的 Google Drive**

同樣把下面這格程式碼**每一行開頭的 `#` 刪掉**，再按 ▶ 執行。

完成後，全部報告會在你的 Google Drive「我的雲端硬碟」裡的 `out_charts` 資料夾。

注意：如果「我的雲端硬碟」裡已經有 `out_charts` 資料夾，這次的結果會被放進它裡面的子資料夾。建議先把舊的資料夾改名或刪除，再執行。


In [ ]:

#cmd2 = 'cp -r /content/out_charts /content/drive/MyDrive/out_charts'
#subprocess.run(cmd2, shell=True)


# **打完收工**